**This notebook is used to show the outcome of extracting text from files**

## Imports

In [1]:
import os
import json
import unicodedata
import xlrd
from openpyxl import load_workbook
from pptx import Presentation
from docx import Document
#from spire.doc import Document as SpireDocument
import requests
import json
#from spire.doc import FileFormat
from sentence_transformers import SentenceTransformer
from bs4 import BeautifulSoup, Tag, NavigableString
import pdfplumber
from lxml import etree
from langchain.text_splitter import RecursiveCharacterTextSplitter
import pandas as pd
import faiss
import pprint
import win32com.client
from operator import itemgetter
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.storage.index_store import SimpleIndexStore
from llama_index.core.storage.kvstore.simple_kvstore import SimpleKVStore

c:\Users\txcjs\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Extraction

### Excel

#### .Xls

In [8]:
simple_xls_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\file_example_XLS_50.xls'
harder_xls_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\financial-data.xls'

In [9]:
def extract_text_from_xls(file_path):
    """Extracts .xls file, ignoring formulas"""
    book = xlrd.open_workbook(file_path)
    all_text = ""

    for sheet in book.sheets():
        all_text += f"--- Sheet: {sheet.name} ---\n"
        for row_idx in range(sheet.nrows):
            row_values = sheet.row_values(row_idx)
            line_parts = []
            for val in row_values:
                if isinstance(val, float):
                    line_parts.append(f"{val:.7g}")  # Prevent floating points stuff from going crazy
                else:
                    line_parts.append(str(val).strip())
            all_text += "\t".join(line_parts) + "\n"
        all_text += "\n"
    
    return all_text

print(extract_text_from_xls(simple_xls_file))

--- Sheet: Sheet1 ---
0	First Name	Last Name	Gender	Country	Age	Date	Id
1	Dulce	Abril	Female	United States	32	15/10/2017	1562
2	Mara	Hashimoto	Female	Great Britain	25	16/08/2016	1582
3	Philip	Gent	Male	France	36	21/05/2015	2587
4	Kathleen	Hanner	Female	United States	25	15/10/2017	3549
5	Nereida	Magwood	Female	United States	58	16/08/2016	2468
6	Gaston	Brumm	Male	United States	24	21/05/2015	2554
7	Etta	Hurn	Female	Great Britain	56	15/10/2017	3598
8	Earlean	Melgar	Female	United States	27	16/08/2016	2456
9	Vincenza	Weiland	Female	United States	40	21/05/2015	6548
10	Fallon	Winward	Female	Great Britain	28	16/08/2016	5486
11	Arcelia	Bouska	Female	Great Britain	39	21/05/2015	1258
12	Franklyn	Unknow	Male	France	38	15/10/2017	2579
13	Sherron	Ascencio	Female	Great Britain	32	16/08/2016	3256
14	Marcel	Zabriskie	Male	Great Britain	26	21/05/2015	2587
15	Kina	Hazelton	Female	Great Britain	31	16/08/2016	3259
16	Shavonne	Pia	Female	France	24	21/05/2015	1546
17	Shavon	Benito	Female	France	39	15/10/2017	

In [10]:
print(extract_text_from_xls(harder_xls_file))

--- Sheet: CapBudgWS ---
Equity Analysis of a Project											
				INPUT SHEET: USER ENTERS ALL BOLD NUMBERS							
INITIAL INVESTMENT				CASHFLOW DETAILS				DISCOUNT RATE			
Initial Investment=		50000		Revenues in  year 1=		40000		Approach(1:Direct;2:CAPM)=		2	
Opportunity cost (if any)=		7484		Var. Expenses as % of Rev=		0.5		1. Discount rate =		0.1	
Lifetime of the investment		10		Fixed expenses in year 1=		0		2a. Beta		0.9	
Salvage Value at end of project=		10000		Tax rate on net income=		0.4		b. Riskless rate=		0.08	
Deprec. method(1:St.line;2:DDB)=		2		If you do not have the breakdown of fixed and variable				c. Market risk premium =		0.055	
Tax Credit (if any )=		0.1		expenses, input the entire expense as a % of revenues.				d. Debt Ratio =		0.3	
Other invest.(non-depreciable)=		0						e. Cost of Borrowing =		0.09	
								Discount rate used=		0.10685	
WORKING CAPITAL											
Initial Investment in Work. Cap=		10000									
Working Capital as % of Rev=		0.25									
Salvag

#### .Xlsx

In [11]:
simple_xlsx_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\file_example_XLSX_50.xlsx'
harder_xlsx_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\financial-data.xlsx'

In [12]:
def extract_text_from_xlsx(file_path):
    """Extracts .xlsx files, ignoring formulas"""
    wb = load_workbook(filename=file_path, data_only=True) # Don't extract formulas
    all_text = ""

    for sheet in wb.sheetnames:
        ws = wb[sheet]
        all_text += f"--- Sheet: {sheet} ---\n"
        for row in ws.iter_rows():
            row_values = []
            for cell in row:
                value = str(cell.value) if cell.value is not None else ""
                row_values.append(value)
            all_text += "\t".join(row_values).rstrip() + "\n"
        all_text += "\n"
    
    return all_text


print(extract_text_from_xlsx(simple_xlsx_file))

--- Sheet: Sheet1 ---
0	First Name	Last Name	Gender	Country	Age	Date	Id
1	Dulce	Abril	Female	United States	32	15/10/2017	1562
2	Mara	Hashimoto	Female	Great Britain	25	16/08/2016	1582
3	Philip	Gent	Male	France	36	21/05/2015	2587
4	Kathleen	Hanner	Female	United States	25	15/10/2017	3549
5	Nereida	Magwood	Female	United States	58	16/08/2016	2468
6	Gaston	Brumm	Male	United States	24	21/05/2015	2554
7	Etta	Hurn	Female	Great Britain	56	15/10/2017	3598
8	Earlean	Melgar	Female	United States	27	16/08/2016	2456
9	Vincenza	Weiland	Female	United States	40	21/05/2015	6548
10	Fallon	Winward	Female	Great Britain	28	16/08/2016	5486
11	Arcelia	Bouska	Female	Great Britain	39	21/05/2015	1258
12	Franklyn	Unknow	Male	France	38	15/10/2017	2579
13	Sherron	Ascencio	Female	Great Britain	32	16/08/2016	3256
14	Marcel	Zabriskie	Male	Great Britain	26	21/05/2015	2587
15	Kina	Hazelton	Female	Great Britain	31	16/08/2016	3259
16	Shavonne	Pia	Female	France	24	21/05/2015	1546
17	Shavon	Benito	Female	France	39	15/10/2017	

In [13]:
print(extract_text_from_xlsx(harder_xlsx_file))

--- Sheet: CapBudgWS ---
Equity Analysis of a Project
				INPUT SHEET: USER ENTERS ALL BOLD NUMBERS
INITIAL INVESTMENT				CASHFLOW DETAILS				DISCOUNT RATE
Initial Investment=		50000		Revenues in  year 1=		40000		Approach(1:Direct;2:CAPM)=		2
Opportunity cost (if any)=		7484		Var. Expenses as % of Rev=		0.5		1. Discount rate =		0.1
Lifetime of the investment		10		Fixed expenses in year 1=		0		2a. Beta		0.9
Salvage Value at end of project=		10000		Tax rate on net income=		0.4		 b. Riskless rate=		0.08
Deprec. method(1:St.line;2:DDB)=		2		If you do not have the breakdown of fixed and variable				 c. Market risk premium =		0.055
Tax Credit (if any )=		0.1		expenses, input the entire expense as a % of revenues.				 d. Debt Ratio =		0.3
Other invest.(non-depreciable)=		0						 e. Cost of Borrowing =		0.09
								Discount rate used=		0.10685
WORKING CAPITAL
Initial Investment in Work. Cap=		10000
Working Capital as % of Rev=		0.25
Salvageable fraction at end=		1

GROWTH RATES
		1	2	3	4	5	6	

### CSV

In [14]:
csv_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\customers-100.csv'

In [15]:
# Btw csv files only have one sheet
def extract_text_from_csv(file_path):
    """Extracts .csv files"""
    df = pd.read_csv(file_path)
    return df.to_string(index=False)

print(extract_text_from_csv(csv_file))

 Index     Customer Id First Name   Last Name                         Company                City                                      Country                Phone 1                Phone 2                              Email Subscription Date                           Website
     1 DD37Cf93aecA6Dc     Sheryl      Baxter                 Rasmussen Group        East Leonard                                        Chile           229.077.5154       397.884.0519x718           zunigavanessa@smith.info         24/8/2020        http://www.stephenson.com/
     2 1Ef7b82A4CAAD10    Preston      Lozano                     Vega-Gentry   East Jimmychester                                     Djibouti             5153435776       686-620-1820x944                    vmata@colon.com         23/4/2021             http://www.hobbs.com/
     3 6F94879bDAfE5a6        Roy       Berry                   Murillo-Perry       Isabelborough                          Antigua and Barbuda                  -1199    (49

### TXT

In [16]:
txt_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\Mission.txt'

In [17]:
def extract_text_from_txt(file_path):
    """Extracts .txt files"""
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()
    
print(extract_text_from_txt(txt_file))

About Us
Verztec is a leader in corporate learning and language solutions.
Committed to enduring success, Verztec believes that true success is measured not only by financial performance but also by the enduring mutual successes cultivated with our clients and partners.
Founded in 2000, Verztec supports a third of the world’s Fortune 500 companies in digital training for their global workforce and localization of business communications. Our esteemed clientele includes IBM, HP, Microsoft, Citibank, Sony, P&G, Emerson, McDonald’s Corporation, UBS, Unilever, and other global brands.
We have collaborated with clients of varying sizes across industries such as Medical, Pharmaceutical and Life Sciences, Petrochemical, Legal, Aerospace, Tourism, Banking and Finance, Legal, IT, and Electronics.
At Verztec, we prioritize innovation and staying ahead of technological advancements. This entails keeping abreast of industry trends, investing in cutting-edge technology, and developing proprietary s

### HTML

In [18]:
html_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\sample1.html'
advanced_html_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\advancedhtml.html'

In [19]:
def extract_text_from_html(file_path):
    """Extract .html files"""
    with open(file_path, 'r', encoding='utf-8') as file:
        soup = BeautifulSoup(file, 'html.parser')

    def format_element(element):
        """
        Apply custom formatting to certain HTML elements:
        - <li>: Format as list item with a dash
        - <table>: Format as a tab-separated grid
        - <a>: Replace with "text (href)" format
        """
        if element.name == 'li':
            return f"- {element.get_text(' ', strip=True)}\n"

        elif element.name == 'table':
            table_text = []
            for row in element.find_all('tr'):
                row_text = []
                for cell in row.find_all(['td', 'th']):
                    # Handles cells in tables
                    cell_text = cell.get_text(" ", strip=True).replace('\r', '').replace('\x07', '')
                    row_text.append(cell_text)
                table_text.append(' | '.join(row_text))
            return '\n'.join(table_text) + '\n'
        
        elif element.name == 'a' and element.has_attr('href'):
            return f"{element.get_text(strip=True)} ({element['href']})"
        else:
            return None

    def traverse(node):
        """Traverse the HTML tree and extract formatted text."""
        output = ''

        for child in node.children:
            if isinstance(child, NavigableString):
                output += child
            elif isinstance(child, Tag):
                if child.name in ['script', 'style']:
                    continue
                # Deal with hyperlinks
                if child.name == 'a' and child.has_attr('href'):
                    output += format_element(child)
                elif child.name in ['li', 'table']:
                    output += format_element(child)
                else:
                    output += traverse(child)
        return output

    body = soup.body or soup  # Fallback if the text is not associated with a <body> thing
    formatted_text = traverse(body)
    return "\n".join(line.strip() for line in formatted_text.splitlines() if line.strip())


print(extract_text_from_html(html_file))

Sample HTML 1
Minime vero, inquit ille, consentit.
Lorem ipsum dolor sit amet, consectetur adipiscing elit. Inscite autem medicinae et gubernationis ultimum cum ultimo sapientiae comparatur. Cur igitur, cum de re conveniat, non malumus usitate loqui?
- Si qua in iis corrigere voluit, deteriora fecit.
- At quicum ioca seria, ut dicitur, quicum arcana, quicum occulta omnia?
- An dolor longissimus quisque miserrimus, voluptatem non optabiliorem diuturnitas facit?
- Multoque hoc melius nos veriusque quam Stoici.
- Stuprata per vim Lucretia a regis filio testata civis se ipsa interemit.
- Ego vero isti, inquam, permitto.
Graecum enim hunc versum nostis omnes-: Suavis laborum est praeteritorum memoria. Qui enim existimabit posse se miserum esse beatus non erit. Si qua in iis corrigere voluit, deteriora fecit. Si qua in iis corrigere voluit, deteriora fecit. (http://loripsum.net/) Dic in quovis conventu te omnia facere, ne doleas. Tu quidem reddes;
- Duo Reges: constructio interrete.
- Contin

In [20]:
print(extract_text_from_html(advanced_html_file))

This is some text before the table. It explains what the table is about.
Band | Year formed | No. of Albums | Most famous song
Buzzcocks | 1976 | 9 | Ever fallen in love
The Clash | 1976 | 6 | London Calling
Total albums | 15
This is some text after the table. It might include a conclusion or next steps.


### Powerpoint

#### .PPT

In [21]:
ppt_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\file_example_PPT_250kB.ppt'

In [22]:
def extract_text_from_ppt(ppt_path):
    """Extract text and speaker notes from .ppt"""
    powerpoint = win32com.client.Dispatch("PowerPoint.Application")
    powerpoint.Visible = 1

    presentation = powerpoint.Presentations.Open(ppt_path, WithWindow=False)
    all_text = []

    for i, slide in enumerate(presentation.Slides, start=1):
        slide_text = [f"--- Slide {i} ---"]

        for shape in slide.Shapes:
            # Handle text and bullet lists
            if shape.HasTextFrame:
                tf = shape.TextFrame
                if tf.HasText:
                    paragraphs = []
                    for paragraph in tf.TextRange.Paragraphs():
                        text = paragraph.Text.strip().replace('\r', '')
                        if text:
                            bullet = "- " if paragraph.ParagraphFormat.Bullet.Type != 0 else ""
                            paragraphs.append(bullet + text)
                    if paragraphs:
                        slide_text.append("\n".join(paragraphs))

            # Handle tables
            if shape.HasTable:
                table = shape.Table
                table_text = []
                for row in range(1, table.Rows.Count + 1):
                    row_text = []
                    for col in range(1, table.Columns.Count + 1):
                        cell = table.Cell(row, col)
                        cell_text = cell.Shape.TextFrame.TextRange.Text.strip().replace('\r', '').replace('\x07', '')
                        row_text.append(cell_text)
                    table_text.append(" | ".join(row_text))
                slide_text.append("\n".join(table_text))

        # Speaker Notes
        if slide.NotesPage.Shapes.Placeholders.Count >= 2:
            notes_shape = slide.NotesPage.Shapes.Placeholders(2)
            if notes_shape.HasTextFrame and notes_shape.TextFrame.HasText:
                notes = notes_shape.TextFrame.TextRange.Text.strip().replace('\r', '')
                if notes:
                    slide_text.append(f"[Notes] {notes}")

        all_text.append("\n".join(slide_text))

    presentation.Close()
    powerpoint.Quit()

    return "\n\n".join(all_text)


print(extract_text_from_ppt(ppt_file))

--- Slide 1 ---
Lorem ipsum
Lorem ipsum dolor sit amet, consectetur adipiscing elit. Nunc ac faucibus odio. Vestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut varius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum condimentum. Vivamus dapibus sodales ex, vitae malesuada ipsum cursus convallis. Maecenas sed egestas nulla, ac condimentum orci. Mauris diam felis, vulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus nisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum, ac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi sit amet tortor quis risus auctor condimentum. Morbi in ullamcorper elit. Nulla iaculis tellus sit amet mauris tempus fringilla.
Maecenas mauris lectus, lobortis et purus mattis, blandit dictum tellus. Maecenas non lorem quis tellus placerat varius. Nulla facilisi. Aenean congue fringilla j

#### .PPTX

In [23]:
pptx_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\Dickinson_Sample_Slides.pptx'

In [24]:
def is_bullet_paragraph(paragraph):
    """
    Check if a paragraph has bullet formatting by inspecting XML.
    We have to do this since python-pptx cannot detect bullets natively :(
    """
    pPr = paragraph._element.pPr
    return pPr is not None and pPr.find(".//a:buChar", namespaces={'a': 'http://schemas.openxmlformats.org/drawingml/2006/main'}) is not None

def extract_text_from_pptx(file_path):
    """Extract text and notes from a .pptx file."""
    presentation = Presentation(file_path)
    all_text = []

    for i, slide in enumerate(presentation.slides, start=1):
        slide_text = [f"--- Slide {i} ---"]

        for shape in slide.shapes:
            # Handle text and bullet lists
            if shape.has_text_frame:
                paragraphs = []
                for para in shape.text_frame.paragraphs:
                    text = para.text.strip()
                    if not text:
                        continue

                    if is_bullet_paragraph(para):
                        indent = "  " * para.level
                        paragraphs.append(f"{indent}- {text}")
                    else:
                        paragraphs.append(text)

                if paragraphs:
                    slide_text.append("\n".join(paragraphs))

            # Handle Tables
            if shape.shape_type == 19:  # MSO_SHAPE_TYPE.TABLE
                table = shape.table
                table_text = []
                for row in table.rows:
                    row_text = []
                    for cell in row.cells:
                        cell_text = cell.text.strip().replace('\r', '').replace('\x07', '')
                        row_text.append(cell_text)
                    table_text.append(' | '.join(row_text))
                slide_text.append('\n'.join(table_text))

        # Speaker Notes
        notes_slide = slide.notes_slide if slide.has_notes_slide else None
        if notes_slide:
            notes_text = notes_slide.notes_text_frame.text.strip()
            if notes_text:
                slide_text.append(f"[Notes] {notes_text}")

        all_text.append("\n".join(slide_text))

    return "\n\n".join(all_text)



print(extract_text_from_pptx(pptx_file))

--- Slide 1 ---
Presentation Title
Author
Department
Date
Location

--- Slide 2 ---
Presentation Title
Author
Department
Date
Location
[Notes] Cover slide option #1

--- Slide 3 ---
Presentation Title
Author
Department
Date
Location
[Notes] Cover slide option #3

--- Slide 4 ---
GraphTitle
Additional Notes - Lorem ipsum dolor sit amet, consectetuer adipiscing elit. Aenean commodo ligula eget dolor. Aenean massa. Cum sociis natoque penatibus et magnis dis parturient montes, nascetur ridiculus mus. Donec quam felis, ultricies nec, pellentesque eu, pretium quis, sem

--- Slide 5 ---
Chart Title
Column A | B | C | D
XXXXXXXX | XX | XX | XX
XXXXXXXX | XX | XX | XX
XXXXXXXX | XX | XX | XX
XXXXXXXX | XX | XX | XX
XXXXXXXX | XX | XX | XX
XXXXXXXX | XX | XX | XX

--- Slide 6 ---
List Title
- Lorem ipsum dolor sit amet
- Aenean commodo ligula eget dolor
- Cum sociis natoque penatibus et magnis dis parturient montes
- Donec quam felis, ultricies nec, pellentesque eu
- Lorem ipsum dolor sit amet, 

### Markdown

In [25]:
md_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\markdown-sample.md'

In [26]:
def extract_text_from_md(file_path):
    """Extract .md file"""
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

print(extract_text_from_md(md_file))

An h1 header

Paragraphs are separated by a blank line.

2nd paragraph. *Italic*, **bold**, and `monospace`. Itemized lists
look like:

  * this one
  * that one
  * the other one

Note that --- not considering the asterisk --- the actual text
content starts at 4-columns in.

> Block quotes are
> written like so.
>
> They can span multiple paragraphs,
> if you like.

Use 3 dashes for an em-dash. Use 2 dashes for ranges (ex., "it's all
in chapters 12--14"). Three dots ... will be converted to an ellipsis.
Unicode is supported. ☺



An h2 header
------------

Here's a numbered list:

 1. first item
 2. second item
 3. third item

Note again how the actual text starts at 4 columns in (4 characters
from the left side). Here's a code sample:

    # Let me re-iterate ...
    for i in 1 .. 10 { do-something(i) }

As you probably guessed, indented 4 spaces. By the way, instead of
indenting the block, you can use delimited blocks, if you like:

~~~
define foobar() {
    print "Welcome to flavor

### Word files

#### .Doc

In [27]:
doc_file_path = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\SampleDOCFile_200kb.doc'
company_doc_file_path = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Data\2_Verztec Webmail and Autoresponder.doc'

In [28]:
def extract_text_from_doc(file_path):
    """Extracts .doc files"""

    # Have to open the file in background because its old -_-
    word = win32com.client.Dispatch("Word.Application")
    word.Visible = False
    doc = word.Documents.Open(file_path)

    full_text = []
    content = doc.Content
    start = content.Start
    end = content.End
    bullets = {'•', '‣', '·', '‧', '–', '-', '*', '', '●', '■', '♦', '\uf0b7', 'o'}

    while start < end:
        current_range = doc.Range(start, start + 1)
        # Extract tables
        if current_range.Tables.Count > 0:
            table = current_range.Tables(1)
            table_text = []
            for row in table.Rows:
                row_text = []
                for cell in row.Cells:
                    cell_text = cell.Range.Text.strip().replace('\r', '').replace('\x07', '')
                    row_text.append(cell_text)
                table_text.append(' | '.join(row_text))
            full_text.append('\n'.join(table_text))
            start = table.Range.End

        # Extract paragraphs
        elif current_range.Paragraphs.Count > 0:
            para_range = current_range.Paragraphs(1).Range
            para_text = para_range.Text.strip().replace('\r', '').replace('\x07', '')
            para_text = unicodedata.normalize("NFKC", para_text) # Normalize to raw text

            if para_text:
                list_format = para_range.ListFormat
                indent = ''

                '''This chunk of code is to deal with microsoft word lists'''
                if list_format.ListType != 0:
                    # Get list indent level and marker
                    level = max(list_format.ListLevelNumber, 1)
                    indent = '    ' * (level - 1)
                    marker = list_format.ListString.strip()

                    # Some markers are invisible or from Wingdings/Symbol font (like '\uf0b7')
                    # These don't render well, so we substitute a standard bullet
                    if not marker or not marker.isprintable() or ord(marker[0]) >= 0xF000:
                        marker = '•'
                    para_text = f"{indent}{marker} {para_text}"

                else:
                    stripped = para_text.lstrip()
                    # Now check if the first character is a bullet point
                    if stripped and stripped[0] in bullets:
                        para_text = f"• {stripped[1:].lstrip()}"

                full_text.append(para_text)

            start = para_range.End
        else:
            start = current_range.End  # Fallback to avoid infinite loop

     # Extract text from shapes
    for shape in doc.Shapes:
        if shape.TextFrame.HasText:
            shape_text = shape.TextFrame.TextRange.Text.strip().replace('\r', '').replace('\x07', '')
            if shape_text:
                full_text.append("[Shape Text] " + shape_text)

    for ishape in doc.InlineShapes:
        if hasattr(ishape, "TextFrame") and ishape.TextFrame.HasText:
            shape_text = ishape.TextFrame.TextRange.Text.strip().replace('\r', '').replace('\x07', '')
            if shape_text:
                full_text.append("[Inline Shape Text] " + shape_text)
                
    doc.Close(False)
    word.Quit()
    return '\n'.join(full_text)

extracted_text = extract_text_from_doc(doc_file_path)
print(extracted_text)

This is Heading1 Text
This is a regular paragraph with the default style of Normal. This is a regular paragraph with the default style of Normal. This is a regular paragraph with the default style of Normal. This is a regular paragraph with the default style of Normal. This is a regular paragraph with the default style of Normal.
This is a Defined Block Style Called BlockStyleTest
This is more Normal text.
This is Heading 2 text
This is more Normal text. This is bold, this is italic, and this is bold italic. This is normal. This is in a defined inline style called InlineStyle. This is normal. This is red text. This is normal.
This block is centered.
This is left-aligned.
• First item of bulleted list.
• Second item of bulleted list.
Second paragraph of second item of bulleted list.
• Third item of bulleted list.
    o First item of third item’s nested list
    o Second item of third item’s nested list
• Fourth and final item of main bulleted list.
This is Normal text.
1. First item of 

In [29]:
print(extract_text_from_doc(company_doc_file_path))

Verztec Webmail
When you are not in the office, you may access Verztec Webmail at this URL:
http://webmail.verztec.com
Userid: your email address
Password: your password
This is useful when you need to check emails from home/ outside to reply to your clients/ suppliers.
Please do check emails regularly when you are accessing Internet at home.
Do take note of there will be email relay problems when you send out. As always, please put yourself as one of the recipients to ensure the email REALLY went out.
Verztec SMTP mail Server for use in office MS Outlook:
Incoming and outgoing: mail.verztec.com
Username: Your email address.
• * Note: If you download mails to your home PC using MS Outlook, you will not be able to download the same emails using your office PC - MS Outlook, so please use Webmail from home to keep a copy of the mails. Note that once you’ve downloaded your emails to your Outlook, it will not keep a copy in the Webmail / Server anymore.
If you have problems sending out emai

#### .Docx

In [18]:
docx_file_path = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\front_end\pipeline\data\raw_data\leave policy.docx"
company_docx_file_path = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Data\SOP checklist transcription projects_270225.docx"

In [21]:
def extract_text_from_docx(file_path):
    """Extract .docx files"""
    doc = Document(file_path)
    full_text = []

    namespaces = {
        'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main',
        'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'
    }

    for element in doc.element.body:
        # Handle paragraph
        if element.tag == etree.QName(namespaces['w'], 'p'):
            is_list = element.find('.//w:numPr', namespaces) is not None
            para_text_parts = []

            for child in element:
                # Handle hyperlinks
                if child.tag == etree.QName(namespaces['w'], 'hyperlink'):
                    r_id = child.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
                    link_text = ''
                    for r in child.findall('.//w:t', namespaces):
                        if r.text:
                            link_text += r.text
                    url = doc.part.rels[r_id].target_ref if r_id and r_id in doc.part.rels else ''
                    para_text_parts.append(f"[{link_text}]({url})" if url else link_text)

                # Handle plain text runs
                elif child.tag == etree.QName(namespaces['w'], 'r'):
                    texts = child.findall('.//w:t', namespaces)
                    for t in texts:
                        if t.text:
                            para_text_parts.append(t.text)

            full_line = ''.join(para_text_parts).strip()
            if full_line:
                if is_list:
                    full_line = f"• {full_line}"
                full_text.append(full_line)

        # Handle tables
        elif element.tag == etree.QName(namespaces['w'], 'tbl'):
            for row in element.findall('.//w:tr', namespaces):
                cells = row.findall('.//w:tc', namespaces)
                row_text = []
                for cell in cells:
                    texts = cell.findall('.//w:t', namespaces)
                    cell_text = ''.join(t.text for t in texts if t.text).strip()
                    row_text.append(cell_text)
                full_text.append('| ' + ' | '.join(row_text) + ' |')

    return '\n'.join(full_text)

extracted_text = extract_text_from_docx(docx_file_path)
print(extract_text_from_docx(docx_file_path))

Company Leave Policy:
All leave must be supported with a VALID reason. Please inform your supervisor or Manager of your leave date(s) before applying via e-Leave.
Please key in the reason before submitting your leave application.
Note: Half day leave is from 9am -1pm (AM) or 2pm – 6pm(PM).
**Minimum period to apply leave is half a day (0.5 day).
(1) No paid leave/ sick leave for probationary period (usually first 3 months). For unpaid leave taken after payroll is processed, it will be deducted in the next month's pay.
(2) Leave application to be submitted at least 1 week in advance for approval.
For leave application of 1 week and above, please apply your leave 3 months (or earlier) in advance so that it can be approved/ disapproved.
This is for capacity planning as if too many members in the team are going on leave together, the other members will not be allowed to go on leave during those period.
Also, if you’re not coming to office due to sick leave, urgent leave etc, please inform 

In [32]:
print(extract_text_from_docx(company_docx_file_path))

Transcription Quality & Accuracy Checklist
1. Pre-Transcription Assessment
• Check audio quality: Background noise, distortions, low volume?
• Identify number of speakers & their clarity (accents, speech pace, overlapping)
• Determine if there’s industry-specific jargon, slang, or acronyms.
• Review any provided reference materials (glossaries, style guides, previous transcripts).
2. Handling Unclear or Inaudible Parts
• Use [inaudible hh:mm:ss] for completely unintelligible sections.
• If uncertain, provide the best guess with (?) (e.g., "data migration (?)").
• Timestamp difficult sections for easier review (e.g., [unclear 00:02:15]).
• If multiple speakers are unclear, use [Speaker A], [Speaker B], etc.
3. Context & Consistency Checks
• Does the transcription make logical sense in context?
• Are repeated terms consistent (e.g., names, key terms, abbreviations)?
• Cross-check unclear words with any available resources (e.g., company website, industry articles).
4. Formatting & Compli

### PDF

In [33]:
pdf_file_path = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\tests\sample_files\ast_sci_data_tables_sample.pdf"
company_pdf_file_path = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\data\raw_data\18_Verztec_Pantry Rules_060715.pdf'

In [34]:
def extract_text_from_pdf(file_path):
    """Extracts .pdf files"""
    output = []

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            blocks = []

            # Extract all tables with bbox
            tables = page.find_tables()
            for table in tables:
                table_bbox = table.bbox
                table_content = []
                for row in table.extract():
                    row_text = " | ".join(cell.strip() if cell else "" for cell in row)
                    table_content.append(f"| {row_text} |")
                blocks.append({
                    'type': 'table',
                    'top': table_bbox[1],
                    'bottom': table_bbox[3],
                    'content': "\n".join(table_content)
                })

            # Extract all words
            words = page.extract_words()
            # Group words into lines by their vertical position (rounded)
            lines_map = {}
            for word in words:
                top = round(word['top'], 1)
                if top not in lines_map:
                    lines_map[top] = []
                lines_map[top].append(word)

            # Convert lines_map to list of text blocks
            for top, word_group in lines_map.items():
                line_text = " ".join(w['text'] for w in sorted(word_group, key=lambda w: w['x0']))
                # Check if line overlaps any table
                in_table = False
                for t in blocks:
                    if t['type'] == 'table' and t['top'] <= top <= t['bottom']:
                        in_table = True
                        break
                if not in_table:
                    blocks.append({
                        'type': 'text',
                        'top': top,
                        'content': line_text
                    })

            # Sort blocks by Y position
            blocks_sorted = sorted(blocks, key=itemgetter('top'))

            page_output = [block['content'] for block in blocks_sorted]
            output.append("\n".join(page_output))

    return "\n\n".join(output)


extracted_text = extract_text_from_pdf(company_pdf_file_path)
print(extract_text_from_pdf(company_pdf_file_path))

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


VERZTEC PANTRY RULES
PLEASE KEEP OUR PANTRY CLEAN & NEAT
1. Be considerate and keep the sink area DRY after
washing. Clean the water dispenser surface if you
spill coffee/ tea on it.
2. DISPOSE any unwanted items and tidy up the
fridge.
3. DISPOSE your left over food, boxes/ plastic
containers OUTSIDE the office. When using the
office’s bin, please close the lid properly after
disposal.
4. CLEAN UP the countertops or tables with a damp
cloth after you had your meal in the office. Be
considerate by tidying up.
5. Please DO NOT cut the fruits or other food items
on the kitchen top but using the available chopping
boards.
Thank you for your co-operation to a clean and
insect free pantry environment!
Management of Verztec


## Data Chunking

In [22]:
from llama_index.core.schema import Document

def split_into_documents(text, chunk_size=1000, chunk_overlap=200, title="Untitled", source="unknown.txt", iso=False):
    """
    Splits text into chunks and returns them as LlamaIndex Document objects with metadata.

    Parameters:
        text (str): The full input text to split.
        chunk_size (int): Max characters per chunk.
        chunk_overlap (int): Characters to overlap between chunks.
        title (str): Title of the source document.
        source (str): Filename of the source document.
        iso (bool): Whether the document is about ISO files.

    Returns:
        List[Document]: Chunked Document objects with metadata.
    """
    splitter = RecursiveCharacterTextSplitter( # Used RecussiveCharacterTextSplitter because it's good at identifying paragraphs and natural sections
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_text(text)
    print(f"Total chunks: {len(chunks)}\n")

    documents = []
    for i, chunk in enumerate(chunks):
        metadata = {
            "chunk": i,
            "title": title,
            "source": source,
            "iso": iso
        }
        doc = Document(text=chunk, metadata=metadata)
        documents.append(doc)

    return documents

documents = split_into_documents(extracted_text)

Total chunks: 11



## Embed and Store the chunks

In [36]:
# The embedding model is chosen based on the Hugging Face MTEB leaderboard:
# https://huggingface.co/spaces/mteb/leaderboard?benchmark_name=MTEB(Multilingual,+v2)

# This model is a subset of the multilingual model "multilingual-e5-large-instruct", which was 4th on the leaderboard.
# We selected "intfloat/e5-large-v2" as it offers the best performance-to-efficiency ratio
# It is faster and more lightweight than the multilingual version, making it suitable for our real-time use case.

In [15]:
# This one is for embeding USER query
Settings.embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-large-v2") # MUST BE SAME AS THE ONE USED FOR INDEXING
embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-large-v2")

# Set Ollama as the default LLM globally
Settings.llm = Ollama(model="llama3.2", request_timeout=120, context_window=4096)

In [33]:
def build_or_append_index(documents, embed_model, persist_dir="../data/Embedded", faiss_path="faiss.index", embedding_dim=1024):
    """
    Create or append to a FAISS + LlamaIndex index.

    Parameters:
        documents (List[Document]): New documents to insert
        embed_model (BaseEmbedding): Embedding model (e.g., HuggingFaceEmbedding)
        persist_dir (str): Directory where LlamaIndex metadata is stored
        faiss_path (str): Filename for FAISS index (within persist_dir)
        embedding_dim (int): Embedding vector size
    """
    faiss_file_path = os.path.join(persist_dir, faiss_path)

    os.makedirs(persist_dir, exist_ok=True)
    

    if os.path.exists(faiss_file_path):
        # Load existing FAISS and LlamaIndex
        print("Loading existing index...")
        faiss_index = faiss.read_index(faiss_file_path)
        vector_store = FaissVectorStore(faiss_index=faiss_index)
        storage_context = StorageContext.from_defaults(
            persist_dir=persist_dir,
            vector_store=vector_store
        )
        index = load_index_from_storage(storage_context)
        print("Total docs in index:", len(index.storage_context.docstore.docs))
        # Append the new documents
        parser = SimpleNodeParser()
        nodes = parser.get_nodes_from_documents(documents)
        index.insert_nodes(nodes)

        # Persist changes
        index.storage_context.persist(persist_dir=persist_dir)
        faiss.write_index(faiss_index, faiss_file_path)

    else:
        # Create new FAISS and LlamaIndex
        print("Creating new index...")
        faiss_index = faiss.IndexFlatL2(embedding_dim)
        vector_store = FaissVectorStore(faiss_index=faiss_index)
        storage_context = StorageContext.from_defaults(
            vector_store=vector_store
        )
        
        kvstore = SimpleKVStore()
        docstore = SimpleDocumentStore(kvstore)
        index_store = SimpleIndexStore(kvstore)

        storage_context = StorageContext.from_defaults(
            docstore=docstore,
            index_store=index_store,
            vector_store=vector_store
        )
        
        index = VectorStoreIndex.from_documents(
            documents, storage_context=storage_context, embed_model=embed_model
        )

    print("Saving new data...")
    index.storage_context.persist(persist_dir=persist_dir)
    faiss.write_index(faiss_index, faiss_file_path)
    print("Total docs in index:", len(index.storage_context.docstore.docs))
    return index


build_or_append_index(documents, embed_model, persist_dir="../data/Embedded", faiss_path="faiss.index", embedding_dim=1024)

Loading existing index...
Total docs in index: 76
Saving new data...
Total docs in index: 87


In [34]:
# ...existing code...
faiss_file_path = os.path.join(persist_dir, faiss_path)
print("FAISS index path:", os.path.abspath(faiss_file_path))
# ...existing code...

FAISS index path: c:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\front_end\pipeline\data\Embedded\faiss.index


In [48]:
extracted_text

'VERZTEC PANTRY RULES\nPLEASE KEEP OUR PANTRY CLEAN & NEAT\n1. Be considerate and keep the sink area DRY after\nwashing. Clean the water dispenser surface if you\nspill coffee/ tea on it.\n2. DISPOSE any unwanted items and tidy up the\nfridge.\n3. DISPOSE your left over food, boxes/ plastic\ncontainers OUTSIDE the office. When using the\noffice’s bin, please close the lid properly after\ndisposal.\n4. CLEAN UP the countertops or tables with a damp\ncloth after you had your meal in the office. Be\nconsiderate by tidying up.\n5. Please DO NOT cut the fruits or other food items\non the kitchen top but using the available chopping\nboards.\nThank you for your co-operation to a clean and\ninsect free pantry environment!\nManagement of Verztec'

In [35]:
# WORKS!
persist_dir = "../data/Embedded"
faiss_path = "faiss.index"
faiss_file_path = os.path.join(persist_dir, faiss_path)

# Load existing FAISS and LlamaIndex
faiss_index = faiss.read_index(faiss_file_path)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(
    persist_dir=persist_dir,
    vector_store=vector_store
)
index = load_index_from_storage(storage_context)
# Sample query for testing
query_engine = index.as_query_engine(similarity_top_k=5)
response = query_engine.query("What is the leave policy?")
pprint.pp(response)

Response(response='The leave policy requires all leave to be supported with a '
                  'valid reason. Applications must be submitted in advance, '
                  "with at least one week's notice for shorter leave periods "
                  "and three months' notice for longer leave periods. The "
                  'policy also includes restrictions on paid and unpaid leave, '
                  'leave encashment, and unused leave being forfeited.',
         source_nodes=[NodeWithScore(node=TextNode(id_='97df5eb9-7c9d-43c9-b602-445faf370e45', embedding=None, metadata={'chunk': 0, 'title': 'Untitled', 'source': 'unknown.txt', 'iso': False}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='e0d78623-e450-4b38-8168-16fe8fc8beda', node_type='4', metadata={'chunk': 0, 'title': 'Untitled', 'source': 'unknown.txt', 'iso': False}, hash='c7a494448d9459beedf48f84a3b3295690e1dae3f7dccefd921db2ef5473

In [17]:
# print one folder above this dir
print(os.path.abspath(os.path.join(os.getcwd(), os.pardir, 'data')))

c:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\front_end\pipeline\data


In [ ]:
# Appends the new documents
new_documents = documents
parser = SimpleNodeParser()
nodes = parser.get_nodes_from_documents(new_documents)
index.insert_nodes(nodes)

# Persist changes
index.storage_context.persist(persist_dir=persist_dir)
faiss.write_index(faiss_index, faiss_file_path)

In [72]:
pprint.pp(Settings.llm)


Ollama(callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x0000024CD05B1780>, system_prompt=None, messages_to_prompt=<function messages_to_prompt at 0x0000024C5D1353F0>, completion_to_prompt=<function default_completion_to_prompt at 0x0000024C5D328790>, output_parser=None, pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'>, query_wrapper_prompt=None, base_url='http://localhost:11434', model='llama3.2', temperature=None, context_window=-1, request_timeout=120.0, prompt_key='prompt', json_mode=False, additional_kwargs={}, is_function_calling_model=True, keep_alive=None)


In [30]:
# Print all documents in the index
for doc_id, doc in index.storage_context.docstore.docs.items():
    print(f"Document ID: {doc_id}")
    print(f"Title: {doc.metadata.get('title', 'Untitled')}")
    print(f"Source: {doc.metadata.get('source', 'unknown.txt')}")
    print(f"Text: {doc.text[:200]}...")  # Print first 200 chars
    print("-" * 40)

Document ID: 953c807c-612e-4b32-bfa2-7de00572fbe1
Title: 11A_Basic Meeting Etiquette for Professionals.pdf
Source: 11A_Basic Meeting Etiquette for Professionals.pdf
Text: Please take note of the below guidelines for all your client / prospect meetings
Basic Meeting Etiquette & Preparation for Professionals
1. Always arrive at Prospect/Client Meetings at least 10 meetin...
----------------------------------------
Document ID: 17e2faa4-e1e9-4a67-ab2d-8a949b1fa05e
Title: 11A_Basic Meeting Etiquette for Professionals.pdf
Source: 11A_Basic Meeting Etiquette for Professionals.pdf
Text: as it may be implied as rather rude.
3. Always stand up and smile warmly when you notice the client(s) has arrived and entering the meeting room. Have a friendly attitude.
4. Please do not pass your n...
----------------------------------------
Document ID: a79d279a-b204-49ac-910c-455225af06bc
Title: 11A_Basic Meeting Etiquette for Professionals.pdf
Source: 11A_Basic Meeting Etiquette for Professionals.pdf
Tex

In [8]:
Settings.llm = Ollama(model="llama3.2", request_timeout=120, context_window=4096) # Set Ollama as the default LLM globally

In [12]:
# Sample query for testing
query_engine = index.as_query_engine(similarity_top_k=5)
response = query_engine.query("What do I do if I want to take a leave?")
pprint.pp(response)

Response(response='To inform your colleagues and management of your intention '
                  'to take a leave, be clear and concise in your email. State '
                  'the specific dates you wish to take off, make sure to give '
                  'enough notice as per company policies, and avoid covering '
                  'multiple topics in the same message. Also, use an '
                  'easy-to-read layout, keep paragraphs short with blank lines '
                  'between each, and number or bullet-point your points if '
                  'necessary.',
         source_nodes=[NodeWithScore(node=TextNode(id_='d11ec0dd-fb6f-4411-a536-6b80f939d387', embedding=None, metadata={'chunk': 1, 'title': '26_Policy on Office Laptop and Computer_050820.pdf', 'source': '26_Policy on Office Laptop and Computer_050820.pdf', 'iso': False}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='623809ea-

## Testing area

In [ ]:
pdf_file_path = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\pipeline\data\raw_data\3_Offboarding Process on Clean Desk Policy_150125.pdf"
daf = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Data\11A_Basic Meeting Etiquette for Professionals.pdf"
def extract_text_from_pdf(file_path):
    """Extracts .pdf files"""
    output = []

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            blocks = []

            # Extract all tables with bbox
            tables = page.find_tables()
            for table in tables:
                table_bbox = table.bbox
                table_content = []
                for row in table.extract():
                    row_text = " | ".join(cell.strip() if cell else "" for cell in row)
                    table_content.append(f"| {row_text} |")
                blocks.append({
                    'type': 'table',
                    'top': table_bbox[1],
                    'bottom': table_bbox[3],
                    'content': "\n".join(table_content)
                })

            # Extract all words
            words = page.extract_words()
            # Group words into lines by their vertical position (rounded)
            lines_map = {}
            for word in words:
                top = round(word['top'], 1)
                if top not in lines_map:
                    lines_map[top] = []
                lines_map[top].append(word)

            # Convert lines_map to list of text blocks
            for top, word_group in lines_map.items():
                line_text = " ".join(w['text'] for w in sorted(word_group, key=lambda w: w['x0']))
                # Check if line overlaps any table
                in_table = False
                for t in blocks:
                    if t['type'] == 'table' and t['top'] <= top <= t['bottom']:
                        in_table = True
                        break
                if not in_table:
                    blocks.append({
                        'type': 'text',
                        'top': top,
                        'content': line_text
                    })

            # Sort blocks by Y position
            blocks_sorted = sorted(blocks, key=itemgetter('top'))

            page_output = [block['content'] for block in blocks_sorted]
            output.append("\n".join(page_output))

    return "\n\n".join(output)

extracted_text = extract_text_from_pdf(daf)
print(extracted_text)
def split_and_print_chunks(text, chunk_size=1000, chunk_overlap=200):
    """
    Splits the input text into chunks and prints each chunk with its index.

    Parameters:
        text (str): The text to be split.
        chunk_size (int): The maximum size of each chunk.
        chunk_overlap (int): The number of overlapping characters between chunks.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_text(extracted_text)


    print(f"Total chunks: {len(chunks)}\n")

    for i, chunk in enumerate(chunks):
        print(f"\n--- Chunk {i + 1} ---\n{chunk}\n{'-' * 60}")

    return chunks
chunks = split_and_print_chunks(extracted_text, chunk_size=1000, chunk_overlap=200)

CropBox missing from /Page, defaulting to MediaBox


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


Please take note of the below guidelines for all your client / prospect meetings
Basic Meeting Etiquette & Preparation for Professionals
1. Always arrive at Prospect/Client Meetings at least 10 meetings before meeting time. Punctuality is very important and goes to show
our personal work attitude. If you know you will be late for the meeting, please ensure you give the client a call to apologize and to
inform them.
2. When you are being ushered into meeting room to wait for clients, please take the seats in the opposite direction from the meeting
room door.
We will want to see and greet clients when they walk into the meeting room, hence do not sit at the seats with our back facing the door
as it may be implied as rather rude.
3. Always stand up and smile warmly when you notice the client(s) has arrived and entering the meeting room. Have a friendly attitude.
4. Please do not pass your name card over the meeting table or wait for the client/prospect to walk up to your seats to greet yo

In [ ]:
file = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\ISO files for whole company\3_COMPANY ORGN CHART.doc"

In [ ]:
def convert_single_doc_to_docx(input_path):
    if not input_path.lower().endswith('.doc') or input_path.lower().endswith('.docx'):
        print("The file must be a .doc file (not .docx).")
        return

    if not os.path.exists(input_path):
        print(f"File not found: {input_path}")
        return

    output_path = os.path.splitext(input_path)[0] + ".docx"
    print(f"Converting {os.path.basename(input_path)} to .docx...")

    try:
        doc = SpireDocument()
        doc.LoadFromFile(input_path)
        doc.SaveToFile(output_path, FileFormat.Docx2016)
        doc.Close()
        print(f"Converted {os.path.basename(input_path)} to .docx successfully.")
    except Exception as e:
        print(f"Failed to convert {input_path} with Spire.Doc: {e}")
convert_single_doc_to_docx(file)

Converting 3_COMPANY ORGN CHART.doc to .docx...
Converted 3_COMPANY ORGN CHART.doc to .docx successfully.


In [ ]:
def convert_doc_to_docx(RAW_DATA):
    for filename in os.listdir(RAW_DATA):
        if filename.lower().endswith('.doc') and not filename.lower().endswith('.docx'):
            input_path = os.path.join(RAW_DATA, filename)
            output_path = os.path.splitext(input_path)[0] + ".docx"

            if not os.path.exists(input_path):
                print(f"File not found: {input_path}")
                continue

            print(f"Converting {filename} to .docx...")

            try:
                doc = SpireDocument()
                doc.LoadFromFile(input_path)
                doc.SaveToFile(output_path, FileFormat.Docx2016)
                doc.Close()
                print(f"Converted {filename} to .docx successfully.")

            except Exception as e:
                print(f"Failed to convert {filename} with Spire.Doc: {e}")

convert_doc_to_docx(file)

NotADirectoryError: [WinError 267] The directory name is invalid: 'C:\\Users\\txcjs\\OneDrive\\Documents\\Homework\\Yr 3.1\\ICP\\ISO files for whole company\\3_COMPANY ORGN CHART.doc'

In [10]:
# Retrieval of supporting documents
retriever = index.as_retriever()
retrieved_nodes = retriever.retrieve("What is the importance of follow up emails?")

for node in retrieved_nodes:
    print(node.text)  # See what would be fed into the LLM


Importance of Customer Follow Ups
Following up is an important aspect in client account management and customer service. It helps
you build trusting ongoing relationships with your clients. And the most significant sales are usually
the result of these relationships.
Like anything worthwhile, consistent follow‐up requires a lot of effort, but in the long run, follow‐ups
are more cost‐effective than acquiring new customers.
There are at least two obvious advantages of the following up process:
Keeping your actual customers.
A good follow‐up with your customers will convince them they’ve made a good choice in selecting to
work with you. Gain their trust and the next time they require any form of services Verztec offers,
they will come to you!
Generate referrals
A satisfied client will surely tell their friends and colleagues about you. Having already experienced
the benefits of your services, he/she will talk about them with first‐hand knowledge and he will get
you valuable referrals.
Ma

In [61]:
# This is streaming

# Define the prompt
data = {
    "model": "llama3.2",
    "prompt": "Why is the sky blue?",
    "stream": True 
}

# Make the streaming request
response = requests.post(
    "http://localhost:11434/api/generate",
    json=data,
    stream=True  # tell requests to stream the response
)

# Stream the response line-by-line
for line in response.iter_lines():
    if line:
        decoded = line.decode("utf-8")
        try:
            content = json.loads(decoded)
            print(content.get("response", ""), end="", flush=True)
        except json.JSONDecodeError:
            pass  # skip malformed chunks


The sky appears blue because of a phenomenon called scattering, which occurs when sunlight interacts with the tiny molecules of gases in the Earth's atmosphere. Here's a simplified explanation:

1. **Sunlight**: When sunlight enters the Earth's atmosphere, it consists of a spectrum of colors, including all the colors of the visible light.
2. **Scattering**: As sunlight travels through the atmosphere, it encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2). These molecules scatter the shorter (blue) wavelengths of light more than the longer (red) wavelengths.
3. **Rayleigh scattering**: The scattering effect is known as Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century. This type of scattering occurs when the light waves encounter small particles or molecules, which scatter the shorter wavelengths more efficiently.
4. **Blue light prevails**: As a result of this scattering, the blue light is scattere

## Previous Solution (For reference)

In [ ]:
def split_and_print_chunks(text, chunk_size=1000, chunk_overlap=200):
    """
    Splits the input text into chunks and prints each chunk with its index.

    Parameters:
        text (str): The text to be split.
        chunk_size (int): The maximum size of each chunk.
        chunk_overlap (int): The number of overlapping characters between chunks.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_text(extracted_text)


    print(f"Total chunks: {len(chunks)}\n")

    for i, chunk in enumerate(chunks):
        print(f"\n--- Chunk {i + 1} ---\n{chunk}\n{'-' * 60}")

    return chunks
chunks = split_and_print_chunks(extracted_text, chunk_size=1000, chunk_overlap=200)

Total chunks: 5


--- Chunk 1 ---
Date: 15th JAN 2025
Offboarding Clean Desk and Digital Handover Policy
This policy outlines the steps employees must take to maintain a clean and organized workspace and ensure
all necessary digital files are properly backed up and handed over before their final day of work.
Clean Desk Policy
Employees must ensure their workspace is clear of personal and unnecessary items by the end of their final
working day. This includes:
1. Removal of Personal Belongings:
o Take home all personal items such as photos, decorations, and personal stationery etc.
o Check and empty all drawers, cabinets, and other storage areas for personal belongings.
o Bring back/ throw away any food or drinks you stored in the office refrigerator.
2. Organizing Work Materials:
o Sort through physical documents. Shred or dispose of sensitive documents no longer
needed. Remove all name cards and old files outside the bin near the lift area.
---------------------------------------------

In [ ]:
def embedding_chunks(file_path, chunks, model):
    """
    Encodes text chunks into embeddings and structures them with metadata.

    Parameters:
        file_path (str): Path to the source file (used for metadata).
        chunks (list): Chunked text data to be embedded.
        model (str): Name of the sentence transformer model to use.

    Returns:
        list: A list of dictionaries containing embeddings and metadata.
    """
    filename = os.path.basename(file_path)
    embeddings = model.encode(chunks)

    vector_data = [
        {
            "id": f"{filename}_chunk{i}",
            "values": embeddings[i].tolist(),
            "metadata": {
                "source": filename,
                "text": chunks[i],
                "chunk": i,
                "chunk_size": len(chunks[i])
            }
        }
        for i in range(len(chunks))
    ]

    print(f"\nEmbedded {len(vector_data)} chunks.\n")

    return vector_data

vector_data = embedding_chunks(pdf_file_path, chunks, model = embed_model)


Embedded 5 chunks.



In [ ]:
vector_data

[{'id': '3_Offboarding Process on Clean Desk Policy_150125.pdf_chunk0',
  'values': [-0.01643088273704052,
   -0.04844304919242859,
   0.018251189962029457,
   0.01643812470138073,
   -0.04652078077197075,
   0.023756926879286766,
   -0.014605456963181496,
   -0.015402242541313171,
   -0.037028566002845764,
   0.04183908924460411,
   0.031195515766739845,
   -0.03887730464339256,
   0.047542549669742584,
   0.006213954649865627,
   0.014450379647314548,
   -0.012569465674459934,
   0.0031801536679267883,
   -0.024809328839182854,
   0.03613077849149704,
   -0.03962685540318489,
   0.01820325292646885,
   -0.04582853615283966,
   0.01671123504638672,
   0.04214471951127052,
   -0.015077177435159683,
   -0.07763653993606567,
   -0.03061138652265072,
   0.006629543844610453,
   -0.043508756905794144,
   0.02561146952211857,
   0.005638137459754944,
   0.008212633430957794,
   0.013132680207490921,
   -0.0545986108481884,
   0.033479515463113785,
   -0.04609157517552376,
   -0.043640118092

In [ ]:
# === Config ===
embed_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\data\clean_data\formatted_text.txt'
filename = os.path.basename(embed_file)
model = SentenceTransformer('BAAI/bge-large-en')  # 1024-dimensional embeddings

# === Load the file ===
with open(embed_file, 'r', encoding='utf-8') as f:
    raw_text = f.read()

# === Chunk using LangChain ===
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_text(raw_text)

# === Embed chunks ===
embeddings = model.encode(chunks)

# === Combine into vector list (ready for Pinecone or saving) ===
vector_data = [
    {
        "id": f"{filename}_chunk{i}",
        "values": embeddings[i].tolist(),
        "metadata": {
            "source": filename,
            "text": chunks[i],
            "chunk": i,
            "chunk_size": len(chunks[i])
        }
    }
    for i in range(len(chunks))
]

# === Optional: Show one entry ===
print(f"\nEmbedded {len(vector_data)} chunks.\n")
print("--- Sample chunk ---")
print(vector_data[0]['metadata']['text'])
print(f"\n--- Embedding length: {len(vector_data[0]['values'])} ---")



✅ Embedded 4 chunks.

--- Sample chunk ---
Date: 15th JAN 2025
Offboarding Clean Desk and Digital Handover Policy
This policy outlines the steps employees must take to maintain a clean and organized workspace and ensure
all necessary digital files are properly backed up and handed over before their final day of work.
Clean Desk Policy
Employees must ensure their workspace is clear of personal and unnecessary items by the end of their final
working day. This includes:
1. Removal of Personal Belongings:
o Take home all personal items such as photos, decorations, and personal stationery etc.
o Check and empty all drawers, cabinets, and other storage areas for personal belongings.
o Bring back/ throw away any food or drinks you stored in the office refrigerator.
2. Organizing Work Materials:
o Sort through physical documents. Shred or dispose of sensitive documents no longer
needed. Remove all name cards and old files outside the bin near the lift area.

--- Embedding length: 1024 ---


In [27]:
vector_data[1]['id']

'formatted_text.txt_1'

In [22]:
for i, item in enumerate(vector_data):
    print(f"\n--- Chunk {i} ---")
    print(f"ID: {item['id']}")
    print(f"Chunk #: {item['metadata']['chunk']}")
    print(f"Source: {item['metadata']['source']}")
    print(f"Text preview: {item['metadata']['text'][:50]}...")  # First 200 chars
    print(f"Embedding length: {len(item['values'])}")
    print(f"Embedding sample: {item['values'][:5]}")  # First 5 values
    print("-" * 60)



--- Chunk 0 ---
ID: formatted_text.txt_0
Chunk #: 0
Source: formatted_text.txt
Text preview: Date: 15th JAN 2025
Offboarding Clean Desk and Dig...
Embedding length: 1024
Embedding sample: [-0.021570194512605667, -0.027368921786546707, -0.011659665033221245, 0.04162769019603729, 0.002682835329324007]
------------------------------------------------------------

--- Chunk 1 ---
ID: formatted_text.txt_1
Chunk #: 1
Source: formatted_text.txt
Text preview: 2. Organizing Work Materials:
o Sort through physi...
Embedding length: 1024
Embedding sample: [0.0013925275998190045, -0.007303130347281694, 0.00020290228712838143, 0.02757474221289158, -0.007039081305265427]
------------------------------------------------------------

--- Chunk 2 ---
ID: formatted_text.txt_2
Chunk #: 2
Source: formatted_text.txt
Text preview: o Transfer all work-related files to the designate...
Embedding length: 1024
Embedding sample: [-0.006819797679781914, -0.012808801606297493, -0.0016777735436335206, 0.0084628984

In [40]:
query = "What should I do to maintain a clean and organized workspace?"
query_vector = model.encode([query])[0].tolist()

result = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

for match in result['matches']:
    print(f"\n--- ID: {match['id']} | Score: {match['score']} ---")
    print(match['metadata']['text'])



--- ID: formatted_text.txt_1 | Score: 0.849586666 ---
2. Organizing Work Materials:
o Sort through physical documents. Shred or dispose of sensitive documents no longer
needed. Remove all name cards and old files outside the bin near the lift area.
o Return all company property (e.g., laptop, keyboard, employment card, medical card, office
keys etc) to the designated person or department.
o Please ensure that sensitive documents are not left on display, which can cause data theft
and leaks.
3. Desk Equipment:
o Do a clean up and leave the desk and its equipment (e.g., monitors, keyboards etc) clean
and functional.
Digital File Handover
Employees must ensure that all relevant digital files are properly backed up and handed over. The following
steps should be completed:
1. Backup Necessary Files:
o Transfer all work-related files to the designated shared drive or cloud storage.
o Ensure files are organized logically in folders for easy access.
2. Handover of Credentials:

--- ID: format